# RAG Selection

The purpose of the RAG selection section is to choose the correct option from the top-20 list, or to respond with "none" if the correct one cannot be found. It returns a JSON-formatted response along with a justification.

Model: `qwen3.5:9b`
Platform: Ollama

Prompt, JSON schema, “none” cases, code validation against candidates

Provide: LLM Results Log, Selections

## Libraries

In [1]:
import json
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ollama import chat

In [2]:
MODEL_NAME = "qwen3.5:9b" 

## Data loading

In [3]:
# catalog loading
catalog = pd.read_csv('../data/clean/catalog_clean.csv', dtype={'product_number': str})
catalog = catalog.drop_duplicates(subset=['product_number']).reset_index(drop=True)
catalog['search_text'] = catalog['search_text'].fillna(catalog['clean_label']).fillna('')

In [4]:
# dev set loading
dev = pd.read_csv('../data/clean/test_set_dev.csv', dtype={'gold_product_number': str})
dev_queries = dev.groupby('clean_query').agg({
    'gold_product_number': lambda x: set(str(p) for p in x),
    'product_label': 'first',      # gold label for inspection
    'area_name': 'first',
    'request': 'first'
}).reset_index()

In [5]:
# hybrid-20 candidates loading
hybrid_rankings = pd.read_csv('../data/clean/hybrid_rrf_top20_dev.csv', dtype={'product_number': str})
hybrid_rankings = hybrid_rankings.rename(columns={'rank': 'hybrid_rank'})

# join with search text
hybrid_candidates = hybrid_rankings.merge(
    catalog[['product_number', 'search_text']],
    on='product_number',
    how='left'
)

In [6]:
# group candidates by query
candidates_by_query = {}
for q, group in hybrid_candidates.groupby('clean_query'):
    candidates_by_query[q] = group[['product_number', 'clean_label', 'search_text']].to_dict('records')
print(f"Catalog unique products: {len(catalog):,}")
print(f"Development unique queries: {len(dev_queries):,}")
print(f"Hybrid candidate rows with search_text: {len(hybrid_candidates):,}")

Catalog unique products: 70,720
Development unique queries: 136
Hybrid candidate rows with search_text: 2,720


## Prompt building

In [11]:
def build_prompt(query, candidates):

    candidate_lines = "\n".join([
        f"- [{c['product_number']}] {c['search_text'][:250]}" 
        for c in candidates
    ])
    
    prompt = f"""
    You are a technical wholesale product specialist.
    Customer inquiry: "{query}"
    Candidate products: {candidate_lines}
    Instructions:
    1. Select the single candidate product that best matches the customer inquiry.
    2. If none of the candidate products are an acceptable match, respond with "none".
    3. Do NOT invent new product numbers. You must choose strictly from the provided candidate codes or "none".
    4. Do NOT output any thinking process (<think> tags) or markdown fences. Output raw JSON only starting directly with {{.
    Return ONLY a JSON object in this format:
    {{
    "product_number": "selected product code or none",
    "clean_label": "selected product title or none",
    "reason": "concise explanation in English (max 20 words)"
    }}
    """
    return prompt

## Test

In [14]:
test_row = dev_queries.iloc[0]
test_q = test_row['clean_query']
test_candidates = candidates_by_query[test_q]

test_prompt = build_prompt(test_q, test_candidates)

t0 = time.time()
response = chat(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': test_prompt}],
    format='json',
    think=False,
    options={
        'temperature': 0,
        'num_predict': 200}
)
elapsed = time.time() - t0

print("Raw content:", repr(response.message.content))

parsed_test = json.loads(response.message.content)

print(f"Query: {test_q}")
print(f"Latency: {elapsed:.2f} s")
print("Response:\n", json.dumps(parsed_test, indent=2, ensure_ascii=False))

Raw content: '{\n"product_number": "5701900001",\n"clean_label": "konepaketti m-cube w-connect 18v maxi1 sisältää abs 18 power -akkuporakoneen, ass 18 1/2\\" compact -akkuiskumutteri-/ruuvinvääntimen, abh 18 compact -akkuporavasaran, aws 18-125e power+ -akkukulmahiomakoneen, afs 18 compact -akkupuukkosahan, kolme 18v",\n"reason": "Candidate includes the requested 18V impact driver (ruuvinvääntin) within a complete tool set."\n}'
Query: 11. akkuruuviväännin – 18 v
Latency: 10.53 s
Response:
 {
  "product_number": "5701900001",
  "clean_label": "konepaketti m-cube w-connect 18v maxi1 sisältää abs 18 power -akkuporakoneen, ass 18 1/2\" compact -akkuiskumutteri-/ruuvinvääntimen, abh 18 compact -akkuporavasaran, aws 18-125e power+ -akkukulmahiomakoneen, afs 18 compact -akkupuukkosahan, kolme 18v",
  "reason": "Candidate includes the requested 18V impact driver (ruuvinvääntin) within a complete tool set."
}


## Running RAG selection on dev set

In [15]:
results = []

print(f"Running LLM selection on {len(dev_queries)} dev queries with {MODEL_NAME}...")

for idx, row in dev_queries.iterrows():
    q = row['clean_query']
    gold_set = row['gold_product_number']
    gold_label = row['product_label']
    candidates = candidates_by_query.get(q, [])
    
    # search mapping for official names and valid codes
    valid_codes = {c['product_number'] for c in candidates}
    label_map = {c['product_number']: c['clean_label'] for c in candidates}
    
    prompt = build_prompt(q, candidates)
    
    t0 = time.time()
    try:
        response = chat(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
            think=False, # thinkmode off
            options={
                'temperature': 0,
                'num_predict': 200
            }
        )
        elapsed = time.time() - t0
        data = json.loads(response.message.content)
        
        chosen_code = str(data.get('product_number', '')).strip()
        reason = str(data.get('reason', '')).strip()
        
        # Guardrail checks
        if chosen_code in valid_codes:
            status = "valid"
            chosen_label = label_map.get(chosen_code, '')
            is_hit = chosen_code in gold_set
        elif chosen_code.lower() == "none":
            status = "none"
            chosen_label = "none"
            is_hit = False
        else:
            status = "hallucination"
            chosen_label = str(data.get('clean_label', ''))
            is_hit = False
            
    except Exception as e:
        elapsed = time.time() - t0
        chosen_code = "error"
        chosen_label = "error"
        reason = str(e)
        status = "error"
        is_hit = False
    results.append({
        'clean_query': q,
        'area_name': row['area_name'],
        'chosen_code': chosen_code,
        'chosen_label': chosen_label,
        'gold_label': gold_label,
        'is_hit': is_hit,
        'status': status,
        'reason': reason,
        'latency_s': elapsed
    })
    
    # print progress every 10 queries or at the end
    if (idx + 1) % 10 == 0 or (idx + 1) == len(dev_queries):
        print(f"Processed {idx + 1}/{len(dev_queries)} queries (latest: {elapsed:.2f} s)...")
        
df_results = pd.DataFrame(results)

print("Finished LLM selection!")

Running LLM selection on 136 dev queries with qwen3.5:9b...
Processed 10/136 queries (latest: 14.77 s)...
Processed 20/136 queries (latest: 18.80 s)...
Processed 30/136 queries (latest: 22.35 s)...
Processed 40/136 queries (latest: 12.24 s)...
Processed 50/136 queries (latest: 28.21 s)...
Processed 60/136 queries (latest: 24.69 s)...
Processed 70/136 queries (latest: 34.75 s)...
Processed 80/136 queries (latest: 29.49 s)...
Processed 90/136 queries (latest: 25.78 s)...
Processed 100/136 queries (latest: 29.81 s)...
Processed 110/136 queries (latest: 27.80 s)...
Processed 120/136 queries (latest: 25.87 s)...
Processed 130/136 queries (latest: 27.04 s)...
Processed 136/136 queries (latest: 28.26 s)...
Finished LLM selection!


## Results

In [16]:
# overall metrics
n = len(df_results)
hit1 = df_results['is_hit'].mean()
none_ratio = (df_results['status'] == 'none').mean()
hallucination_ratio = (df_results['status'] == 'hallucination').mean()
avg_latency_ms = df_results['latency_s'].mean() * 1000

print(f"RESULTS: {MODEL_NAME}")
print(f"Hit@1: {hit1:.4f} ({df_results['is_hit'].sum()}/{n})")
print(f"None ratio: {none_ratio * 100:.1f} %")
print(f"Hallucination ratio: {hallucination_ratio * 100:.1f} %")
print(f"Avg Latency: {avg_latency_ms:.1f} ms / query")

RESULTS: qwen3.5:9b
Hit@1: 0.2279 (31/136)
None ratio: 16.9 %
Hallucination ratio: 16.2 %
Avg Latency: 23096.8 ms / query


In [17]:
# visual inspection table
print("\nSample selections for inspection:")
display(df_results[['clean_query', 'chosen_label', 'gold_label', 'is_hit', 'status', 'reason']].head(10))


Sample selections for inspection:


,clean_query,chosen_label,gold_label,is_hit,status,reason
0,11. akkuruuviväännin – 18 v,konepaketti m-cube w-connect 18v maxi1,"AKKU LI-ION 18V/5,0AH M-CUBE W-CONNECT",False,valid,Candidate includes the requested 18V impact dr...
1,"7. akkuruuviväännin – 10,9 v",akkuruuviväännin speede 2 vde,AKKUPORA BS12-A,False,valid,Product name matches inquiry exactly; voltage ...
2,adblue 10 l,adblue 10 l,ADBLUE 10 L,True,valid,Exact match for 10L AdBlue product specified i...
3,akkuporakone 18v ddf484t makita,akkuiskuporakone dhp489z 18v lxt runko,"AKKUPORAVASARA ABH 18 COMPACT, 2X5.0AH",False,valid,Customer requested DDF484T; closest match is D...
4,"askartelupuukko terä n. 100mm ""mora-tyylinen""",none,PUUKKO HIILITERÄS,False,none,"Customer requests a Mora-style chisel blade, b..."
5,ensiaputarvikkeet haavapyyhe 40 kpl / pakkaus,haavapyyhe 40 kpl 3227,HAAVAPYYHE 40 KPL 3227,True,valid,Matches exact product type and quantity of 40 ...
6,ensiaputarvikkeet kylmähaudepussi 6 kpl / pakkaus,kylmähaude 6 kpl 219600,KYLMÄHAUDE 6 KPL 219600,True,valid,Product matches exact item type and quantity s...
7,ensiaputarvikkeet pelastuspeite,pelastuspeite 1892,PELASTUSPEITE 1892,True,valid,Matches 'pelastuspeite' in inquiry; other cand...
8,ensiaputarvikkeet turvaleikkuri,turvaleikkuri patruunoille muovipatruunoiden l...,TURVAVYÖLEIKKURI,False,hallucination,Product is a safety cutter matching the 'turva...
9,graffitinpoistosuihke,graffitinpoistaja 500 ml,GRAFFITINPOISTAJA 500 ML,True,valid,Product name and description directly match th...


In [19]:
# why did the model hallucinate
display(df_results[df_results['status'] == 'hallucination'][['clean_query', 'chosen_code', 'reason']].head(5))



,clean_query,chosen_code,reason
8,ensiaputarvikkeet turvaleikkuri,[071566 09],Product is a safety cutter matching the 'turva...
11,harjanvarsi 28x1400 kartio puinen,[0715735090],Customer requests a wooden brush handle; this ...
21,"hienomek. ruuvitaltta 2,5x50 -1kpl",[018125 50],Matches customer request for 2.5x50 hex socket...
38,kestopeite 8x12m,[0818000296],Exact match for product name and dimensions 8x...
40,kierrepuunporanterä (koko 6mm),[087993 06],"Product is a 6mm hole saw, matching the Finnis..."


In [20]:
display(df_results[df_results['status'] == 'none'][['clean_query', 'reason']].head(5))

,clean_query,reason
4,"askartelupuukko terä n. 100mm ""mora-tyylinen""","Customer requests a Mora-style chisel blade, b..."
16,henkilön suojaimet / työturvallisuus nitriilik...,"Customer requests general nitrile gloves, but ..."
17,henkilön suojaimet / työturvallisuus suojalasi...,"Customer requests person protection eyewear, b..."
20,henkselit dimex 4476+,No candidate matches the specific Henkel Dimex...
46,konehiomapaperi hiomant no 120,"Customer requests grinding paper, but candidat..."


In [23]:
# comparison with previous reranking results
with open('../data/clean/reranking_results.json', 'r', encoding='utf-8') as f:
    rerank_data = json.load(f)
comparison_df = pd.DataFrame(rerank_data['results'])

llm_summary = {
    'Method': f"LLM Selection ({MODEL_NAME})",
    'Hit@1': round(hit1, 4),
    'Recall@5': np.nan,   # LLM picks only 1 item
    'Recall@10': np.nan,
    'MRR': round(hit1, 4),
    'nDCG@5': np.nan,
    'Latency (ms/query)': round(avg_latency_ms, 1)
}

comparison_df = pd.concat([comparison_df, pd.DataFrame([llm_summary])], ignore_index=True)
display(comparison_df[['Method', 'Hit@1', 'MRR', 'Latency (ms/query)']])

,Method,Hit@1,MRR,Latency (ms/query)
0,Hybrid RRF (k=60),0.227941,0.307379,3.570683
1,Cross-Encoder (bge-reranker-v2-m3),0.330882,0.413086,1248.296506
2,LLM Selection (qwen3.5:9b),0.227900,0.227900,23096.800000


The cross-encoder yielded the best accuracy. The LLM was overly cautious and hallucinated in more than 16% of cases (invented the code itself), so it is not a viable solution as it stands. This may have been due to the number of candidates being too large at once; furthermore, the quality of the candidates should be improved. Consequently, one-third of all queries failed.

Garbage in, garbage out.

With this data, the hybrid model’s Recall@20 is estimated to be around 55–60%.

This means that in as many as 40% of the queries, the correct product was not even on the provided list!

The encoder performed better because it strictly compares only one pair at a time: (query, item 1), (query, item 2).

**Recommendation:**

In the future, it would be worth chaining this:

[Hybrid BM25 + Dense] -> Top 100 -> [Rerank: Cross-Encoder] -> Top 5-3 -> [LLM Selection]



## Saving

In [24]:
# save detailed selection log to CSV
log_path = f'../data/clean/rag_selection_log_{MODEL_NAME.replace(":", "_")}.csv'
df_results.to_csv(log_path, index=False)
print(f"Detailed selection log saved to '{log_path}'")

Detailed selection log saved to '../data/clean/rag_selection_log_qwen3.5_9b.csv'


In [25]:
# category-wise Hit@1 summary
top_cats = dev_queries['area_name'].value_counts().head(5).index
cat_summary = df_results[df_results['area_name'].isin(top_cats)].groupby('area_name')['is_hit'].mean().round(3)
print("\nHit@1 by category:")
print(cat_summary)


Hit@1 by category:
area_name
Kiinnitystarvikkeet                        0.000
Käsityökalut                               0.226
Materiaalin mekaaninen työstö              0.190
Tekniset kemikaalit/-tarvikkeet, teipit    0.214
Työturvallisuus                            0.179
Name: is_hit, dtype: float64


In [26]:
# save summary results to JSON
output_json_path = '../data/clean/rag_selection_results.json'

export_data = {
    'model_name': MODEL_NAME,
    'candidate_source': 'Hybrid RRF (k=60) top 20',
    'results': comparison_df.to_dict(orient='records'),
    'hit1': float(hit1),
    'none_ratio': float(none_ratio),
    'hallucination_ratio': float(hallucination_ratio),
    'latency_ms': float(avg_latency_ms)
}

with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)
print(f"Summary results saved to '{output_json_path}'")

Summary results saved to '../data/clean/rag_selection_results.json'
